# 🏗️ Lakehouse Playground — Setup & Hello World

This notebook initializes the Lakehouse environment and runs your first queries.

Since this notebook runs **inside Docker Compose**, all services are reachable by hostname.

---
## ⚙️ Step 1 — Configuration

In [1]:
import boto3
import requests
import json
import time
from botocore.client import Config

# --- S3 (SeaweedFS) ---
S3_ENDPOINT = "http://seaweedfs:8333"
S3_ACCESS_KEY = "lakehouse-admin"
S3_SECRET_KEY = "lakehouse-secret-key"
S3_REGION = "us-east-1"
S3_BUCKET = "lakehouse"

# --- Polaris (both REST and Management APIs are on port 8181) ---
POLARIS_URL = "http://polaris:8181"
POLARIS_CLIENT_ID = "root"
POLARIS_CLIENT_SECRET = "polaris-secret"

# --- Trino ---
TRINO_HOST = "trino"
TRINO_PORT = 8080

# --- Medallion layers ---
NAMESPACES = ["bronze", "silver", "gold"]

print("✅ Configuration loaded")

✅ Configuration loaded


---
## 🔍 Step 2 — Wait for Services

In [2]:
def wait_for_service(name, url, max_attempts=30):
    """Wait for a service to be ready."""
    print(f"⏳ Waiting for {name}...", end="")
    for attempt in range(max_attempts):
        try:
            r = requests.get(url, timeout=2)
            # Any HTTP response (even 4xx/5xx) means the service is up
            print(f" ✅ {name} is ready! (HTTP {r.status_code})")
            return True
        except requests.ConnectionError:
            pass
        except requests.Timeout:
            pass
        print(".", end="", flush=True)
        time.sleep(2)
    print(f" ❌ {name} did not become ready.")
    return False

wait_for_service("SeaweedFS S3", S3_ENDPOINT)
wait_for_service("Polaris REST API", f"{POLARIS_URL}/api/catalog/v1/config")
wait_for_service("Trino", f"http://{TRINO_HOST}:{TRINO_PORT}/v1/info")

⏳ Waiting for SeaweedFS S3... ✅ SeaweedFS S3 is ready! (HTTP 403)
⏳ Waiting for Polaris REST API... ✅ Polaris REST API is ready! (HTTP 401)
⏳ Waiting for Trino... ✅ Trino is ready! (HTTP 200)


True

---
## 🪣 Step 3 — Create S3 Bucket

In [3]:
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name=S3_REGION,
    config=Config(signature_version="s3v4"),
)

existing = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]

if S3_BUCKET in existing:
    print(f"⚠️  Bucket '{S3_BUCKET}' already exists — skipping")
else:
    s3.create_bucket(Bucket=S3_BUCKET)
    print(f"✅ Created bucket: {S3_BUCKET}")

print(f"\n📋 Bucket ready: 🪣 {S3_BUCKET}")
for ns in NAMESPACES:
    print(f"   └── s3://{S3_BUCKET}/{ns}/")

✅ Created bucket: lakehouse

📋 Bucket ready: 🪣 lakehouse
   └── s3://lakehouse/bronze/
   └── s3://lakehouse/silver/
   └── s3://lakehouse/gold/


---
## 🔑 Step 4 — Get Polaris API Token

In [4]:
token_resp = requests.post(
    f"{POLARIS_URL}/api/catalog/v1/oauth/tokens",
    data={
        "grant_type": "client_credentials",
        "client_id": POLARIS_CLIENT_ID,
        "client_secret": POLARIS_CLIENT_SECRET,
        "scope": "PRINCIPAL_ROLE:ALL",
    },
)

if token_resp.ok:
    POLARIS_TOKEN = token_resp.json()["access_token"]
    headers = {"Authorization": f"Bearer {POLARIS_TOKEN}", "Content-Type": "application/json"}
    print(f"✅ Polaris token obtained (expires in {token_resp.json().get('expires_in', '?')}s)")
else:
    print(f"❌ Failed to get token: {token_resp.status_code} — {token_resp.text}")

✅ Polaris token obtained (expires in 3600s)


---
## 📚 Step 5 — Register Polaris Catalog

In [6]:
catalog_payload = {
    "catalog": {
        "name": "lakehouse",
        "type": "INTERNAL",
        "properties": {
            "default-base-location": f"s3://{S3_BUCKET}",
            "s3.endpoint": S3_ENDPOINT,
            "s3.path-style-access": "true",
            "s3.region": S3_REGION,
            "s3.access-key-id": S3_ACCESS_KEY,
            "s3.secret-access-key": S3_SECRET_KEY
        },
        "storageConfigInfo": {
            "storageType": "S3",
            "stsUnavailable": True,
            "pathStyleAccess": True,
            "endpoint": S3_ENDPOINT,
            "region": S3_REGION,
            "allowedLocations": [
                f"s3://{S3_BUCKET}/"
            ]
        }
    }
}

r = requests.post(f"{POLARIS_URL}/api/management/v1/catalogs", headers=headers, json=catalog_payload)

if r.status_code in (200, 201):
    print("✅ Catalog 'lakehouse' registered (SeaweedFS + path-style)")
elif r.status_code == 409:
    print("⚠️  Catalog 'lakehouse' already exists — skipping")
else:
    print(f"❌ Catalog registration failed: {r.status_code} — {r.text}")

✅ Catalog 'lakehouse' registered (SeaweedFS + path-style)


## 🏷️ Step 6 — Create Medallion Namespaces

In [10]:
for ns in NAMESPACES:
    payload = {
        "namespace": [ns],
        "properties": {}
    }
    r = requests.post(
        f"{POLARIS_URL}/api/catalog/v1/lakehouse/namespaces",
        headers=headers,
        json=payload,
    )
    if r.status_code in (200, 201):
        print(f"✅ Namespace '{ns}' created → s3://{S3_BUCKET}/{ns}/")
    elif r.status_code == 409:
        print(f"⚠️  Namespace '{ns}' already exists — skipping")
    else:
        print(f"❌ Namespace '{ns}' failed: {r.status_code} — {r.text}")

print("\n🎉 Setup complete!")

✅ Namespace 'bronze' created → s3://lakehouse/bronze/
✅ Namespace 'silver' created → s3://lakehouse/silver/
✅ Namespace 'gold' created → s3://lakehouse/gold/

🎉 Setup complete!


---
# 🚀 Step 7 — E-Commerce Data Pipeline with Trino

With the infrastructure ready, let's build a **Medallion Architecture** pipeline using real-world e-commerce order data:

| Layer | Table | Purpose |
|-------|-------|---------|
| 🥉 Bronze | `raw_orders` | Raw JSON events, exactly as ingested |
| 🥈 Silver | `cleansed_orders` | Parsed, typed, and deduplicated |
| 🥇 Gold | `daily_sales_summary` | Business-ready aggregations |

### 🔌 Connect to Trino

In [ ]:
from trino.dbapi import connect

conn = connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user="admin",
    catalog="iceberg",
    schema="bronze",
)
cursor = conn.cursor()

def run_query(sql, display=True):
    """Execute a query and return results."""
    cursor.execute(sql)
    try:
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        if display and rows:
            widths = [max(len(str(c)), max(len(str(r[i])) for r in rows)) for i, c in enumerate(columns)]
            header = " | ".join(c.ljust(w) for c, w in zip(columns, widths))
            sep = "-+-".join("-" * w for w in widths)
            print(header)
            print(sep)
            for row in rows:
                print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
        return rows
    except Exception:
        return []

print("✅ Connected to Trino")

### 📋 Verify Medallion Schemas

In [ ]:
run_query("SHOW SCHEMAS FROM iceberg")

---
## 7.1 — Bronze Layer 🥉 (Raw Ingestion)

The Bronze layer captures **raw, unprocessed events** exactly as they arrive from the source system. Each row stores the original JSON payload alongside ingestion metadata — no parsing, no cleaning, just an immutable audit trail.

In [ ]:
run_query("""
CREATE TABLE IF NOT EXISTS iceberg.bronze.raw_orders (
    event_id    VARCHAR,
    payload     VARCHAR,
    ingest_time TIMESTAMP(6) WITH TIME ZONE
) WITH (format = 'PARQUET')
""")
print("✅ Table 'raw_orders' created in bronze layer")

### 📥 Ingest Raw Order Events

Simulating a stream of order events — notice that `evt5` is a **duplicate** of `evt1` (same `order_id`), which is realistic for at-least-once delivery systems.

In [ ]:
run_query("""
INSERT INTO iceberg.bronze.raw_orders VALUES
    ('evt1', '{"order_id": 101, "customer_id": "C001", "amount": 25.50,  "status": "completed", "order_date": "2026-02-20"}', CURRENT_TIMESTAMP),
    ('evt2', '{"order_id": 102, "customer_id": "C002", "amount": 149.99, "status": "completed", "order_date": "2026-02-20"}', CURRENT_TIMESTAMP),
    ('evt3', '{"order_id": 103, "customer_id": "C001", "amount": 12.00,  "status": "completed", "order_date": "2026-02-21"}', CURRENT_TIMESTAMP),
    ('evt4', '{"order_id": 104, "customer_id": "C003", "amount": 89.90,  "status": "completed", "order_date": "2026-02-21"}', CURRENT_TIMESTAMP),
    ('evt5', '{"order_id": 101, "customer_id": "C001", "amount": 25.50,  "status": "completed", "order_date": "2026-02-20"}', CURRENT_TIMESTAMP)
""")
print("✅ 5 raw events ingested (including 1 duplicate)")

### 🔍 Query Bronze Layer

In [ ]:
run_query("SELECT * FROM iceberg.bronze.raw_orders ORDER BY event_id")

---
## 7.2 — Silver Layer 🥈 (Cleaned & Typed)

The Silver layer provides a **reliable, single source of truth**. Here we:
1. **Parse** JSON fields into proper SQL columns
2. **Cast** values to their correct data types
3. **Deduplicate** using `ROW_NUMBER()` — keeping only the latest event per `order_id`

In [ ]:
run_query("""
CREATE TABLE IF NOT EXISTS iceberg.silver.cleansed_orders (
    order_id    BIGINT,
    customer_id VARCHAR,
    amount      DOUBLE,
    status      VARCHAR,
    order_date  DATE,
    updated_at  TIMESTAMP(6) WITH TIME ZONE
) WITH (format = 'PARQUET')
""")
print("✅ Table 'cleansed_orders' created in silver layer")

### 📥 Transform: Bronze → Silver

In [ ]:
run_query("""
INSERT INTO iceberg.silver.cleansed_orders
SELECT
    CAST(json_extract_scalar(payload, '$.order_id') AS BIGINT),
    json_extract_scalar(payload, '$.customer_id'),
    CAST(json_extract_scalar(payload, '$.amount') AS DOUBLE),
    json_extract_scalar(payload, '$.status'),
    CAST(json_extract_scalar(payload, '$.order_date') AS DATE),
    CURRENT_TIMESTAMP
FROM (
    SELECT payload,
           ROW_NUMBER() OVER (
               PARTITION BY json_extract_scalar(payload, '$.order_id')
               ORDER BY ingest_time DESC
           ) AS rn
    FROM iceberg.bronze.raw_orders
)
WHERE rn = 1
""")
print("✅ Silver layer populated — deduplicated from 5 → 4 unique orders")

### 🔍 Query Silver Layer

In [ ]:
run_query("SELECT * FROM iceberg.silver.cleansed_orders ORDER BY order_id")

---
## 7.3 — Gold Layer 🥇 (Business Data Product)

The Gold layer produces **highly curated, business-ready data products**. This daily sales summary is optimized for BI dashboards and reporting — analysts can query it directly without touching raw data.

In [ ]:
run_query("""
CREATE TABLE IF NOT EXISTS iceberg.gold.daily_sales_summary (
    order_date       DATE,
    total_orders     BIGINT,
    total_revenue    DOUBLE,
    unique_customers BIGINT
) WITH (format = 'PARQUET')
""")
print("✅ Table 'daily_sales_summary' created in gold layer")

### 📥 Aggregate: Silver → Gold

In [ ]:
run_query("""
INSERT INTO iceberg.gold.daily_sales_summary
SELECT
    order_date,
    COUNT(order_id)          AS total_orders,
    SUM(amount)              AS total_revenue,
    COUNT(DISTINCT customer_id) AS unique_customers
FROM iceberg.silver.cleansed_orders
GROUP BY order_date
""")
print("✅ Gold data product generated")

### 🔍 Query Gold Layer

In [ ]:
run_query("SELECT * FROM iceberg.gold.daily_sales_summary ORDER BY order_date")

---
## 🧊 Iceberg Table Metadata

One of Iceberg's superpowers is **time travel** — every write creates an immutable snapshot. Let's inspect the snapshot history of our bronze table.

In [ ]:
run_query('SELECT * FROM iceberg.bronze."raw_orders$snapshots"')

---
## 🧹 Cleanup (Optional)

Drop all tables to start fresh.

In [ ]:
# Uncomment to drop all tables:
# run_query("DROP TABLE IF EXISTS iceberg.gold.daily_sales_summary")
# run_query("DROP TABLE IF EXISTS iceberg.silver.cleansed_orders")
# run_query("DROP TABLE IF EXISTS iceberg.bronze.raw_orders")
# print("🗑️ All tables dropped")